<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
print('DataFrame Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nNull Counts:')
print(df.isnull().sum())
print('\nNumber of exact duplicate rows:', df.duplicated().sum())

DataFrame Shape: (8, 6)

Data Types:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null Counts:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Number of exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [6]:
original_price_type = clean['price'].dtype
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float

# Log the number of rows affected if the data type of the 'price' column actually changed.
# A type conversion for a column affects all rows in that column.
if original_price_type != clean['price'].dtype:
    rows_affected = len(clean)
else:
    rows_affected = 0
log('price', 'stripped dollar signs, whitespace, and converted to float', rows_affected)

[price] stripped dollar signs, whitespace, and converted to float (0 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [7]:
# Coerce qty to numeric (this implicitly handles 'NULL' -> NaN and ensures numerical type)
# In this specific dataset, 'qty' is already float64, and 'NULL' is already NaN, so this line will not change the column values or dtype.
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

# Count of NaN quantities before dropping
missing = clean['qty'].isnull().sum()

# Count of negative quantities before dropping
negative = (clean['qty'] < 0).sum()

# Decision 1: Handle missing quantities
if missing > 0:
    clean = clean.dropna(subset=['qty']).copy()
    log('qty', 'dropped rows with missing quantity', missing)
else:
    log('qty', 'no rows with missing quantity to drop', 0)

# Decision 2: Handle negative quantities (refunds)
# Note: `negative` count was taken before any drops in this step, so it reflects the original count for this step.
if negative > 0:
    clean = clean[clean['qty'] >= 0].copy()
    log('qty', 'dropped rows with negative quantity (refunds)', negative)
else:
    log('qty', 'no rows with negative quantity to drop', 0)

[qty] dropped rows with missing quantity (1 row(s))
[qty] dropped rows with negative quantity (refunds) (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [8]:
print('before:', sorted(clean['category'].unique()))

initial_distinct_categories = clean['category'].nunique()

# Normalize case and remove punctuation
clean['category'] = clean['category'].astype(str).str.lower().str.replace('[^a-z\s]', '', regex=True).str.strip()

# Define mapping for judgment calls
CATEGORY_MAP = {
    'food': 'food',
    'apparel': 'apparel',
    'raingear': 'raingear', # Combining 'RainGear' and 'rain-gear' if both were present
    'merch': 'merch'
}

# Apply the mapping
clean['category'] = clean['category'].replace(CATEGORY_MAP)

final_distinct_categories = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))
log('category', 'normalized categories and mapped variants', initial_distinct_categories - final_distinct_categories)

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']
[category] normalized categories and mapped variants (1 row(s))


<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1153/1732690831.py:6: SyntaxWarning: invalid escape sequence '\s'
  clean['category'] = clean['category'].astype(str).str.lower().str.replace('[^a-z\s]', '', regex=True).str.strip()


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [10]:
print('before:', sorted([x for x in clean['item'].unique().tolist() if pd.notna(x)]))

initial_distinct_items = clean['item'].nunique()

# Decision: Handle missing item names (NaN)
missing_items = clean['item'].isnull().sum()
if missing_items > 0:
    clean = clean.dropna(subset=['item']).copy()
    log('item', 'dropped rows with missing item name', missing_items)
else:
    log('item', 'no rows with missing item name to drop', 0)

# Normalize case, strip whitespace, and remove punctuation
clean['item'] = clean['item'].astype(str).str.lower().str.replace('[^a-z\\s]', '', regex=True).str.strip()

# Define mapping for item names
ITEM_MAP = {
    'cheese burger': 'cheeseburger'
}

# Apply the mapping
clean['item'] = clean['item'].replace(ITEM_MAP)

final_distinct_items = clean['item'].nunique()

print('after: ', sorted(clean['item'].unique().tolist()))
log('item', 'normalized item names and mapped variants', initial_distinct_items - final_distinct_items)

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
[item] dropped rows with missing item name (1 row(s))
after:  ['cheeseburger', 'rain poncho', 'uva tshirt']
[item] normalized item names and mapped variants (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [11]:
original_ts_values = clean['ts'].copy()
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed_conversions = clean['ts'].isnull().sum()
log('ts', 'parsed timestamps and coerced failures to NaT', failed_conversions)

clean['hour'] = clean['ts'].dt.hour

[ts] parsed timestamps and coerced failures to NaT (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [13]:
# Assertions to catch regressions
assert clean.shape == (4, 7), "Expected 4 rows and 7 columns after cleaning"
assert clean['price'].dtype == float, "'price' column should be float"
assert clean['qty'].dtype == float, "'qty' column should be float"
assert clean['ts'].dtype == 'datetime64[ns]', "'ts' column should be datetime64[ns]"
assert clean['hour'].dtype == float, "'hour' column should be float"

assert clean['qty'].isnull().sum() == 0, "'qty' column should not have nulls"
assert clean['item'].isnull().sum() == 0, "'item' column should not have nulls"
assert clean['category'].isnull().sum() == 0, "'category' column should not have nulls"
assert (clean['qty'] >= 0).all(), "'qty' column should not have negative values"
assert sorted(clean['category'].unique()) == ['apparel', 'food', 'raingear'], "Unexpected categories"
assert sorted(clean['item'].unique()) == ['cheeseburger', 'rain poncho', 'uva tshirt'], "Unexpected item names"

# Compute revenue
clean['revenue'] = clean['qty'] * clean['price']

# Print totals
print('Rows after cleaning:', len(clean))
print('Total units sold:', clean['qty'].sum())
print('Total revenue:', clean['revenue'].sum())
print('Distinct categories:', clean['category'].nunique())

Rows after cleaning: 4
Total units sold: 9.0
Total revenue: 94.5
Distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [15]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,"stripped dollar signs, whitespace, and convert...",0
2,qty,dropped rows with missing quantity,1
3,qty,dropped rows with negative quantity (refunds),1
4,category,normalized categories and mapped variants,1
5,item,dropped rows with missing item name,1
6,item,normalized item names and mapped variants,1
7,ts,parsed timestamps and coerced failures to NaT,3


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [16]:
# Checkpoint
rows_after = 4
revenue_after = 94.5
biggest_decision = 'Dropped rows with negative quantity (refunds)'
revenue_other_way = 76.5

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 4
revenue: 94.5
decision that mattered: Dropped rows with negative quantity (refunds)
revenue the other way: 76.5
